In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from sentence_transformers import SentenceTransformer


In [ ]:
df = pd.read_parquet("rag_qa_ptbr_doc_questions.parquet")
df["context_len"] = df["context"].fillna("").astype(str).str.len()


In [ ]:
CHUNK_SIZE = 1024
df_split = df[(df["context_len"] > 500)].copy()
print(len(df_split))
def split_text_into_chunks(text, chunk_size=1024):
    text = "" if pd.isna(text) else str(text)
    return [
        text[i:i + chunk_size]
        for i in range(0, len(text), chunk_size)
        if text[i:i + chunk_size].strip()
    ]

chunk_rows = []

for _, row in tqdm(df_split.iterrows(), total=len(df_split), desc="Chunking"):
    document_id = row["document_id"]
    context = row["context"]

    chunks = split_text_into_chunks(context, CHUNK_SIZE)

    for chunk_idx, chunk_text in enumerate(chunks):
        chunk_rows.append({
            "chunk_id": f"{document_id}_chunk_{chunk_idx}",
            "document_id": document_id,
            "chunk_index": chunk_idx,
            "chunk_text": chunk_text,
            "chunk_start_char": chunk_idx * CHUNK_SIZE,
            "chunk_end_char": chunk_idx * CHUNK_SIZE + len(chunk_text),
        })

chunks_df = pd.DataFrame(chunk_rows)
print(len(chunks_df))

In [ ]:
chunks_df.head()

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
MODEL_NAME = "Octen/Octen-Embedding-0.6B"

model = SentenceTransformer(
    MODEL_NAME,
    model_kwargs={
        "torch_dtype": torch.bfloat16,  # <-- This will remove the warning
        "device_map": "auto",
    },
    tokenizer_kwargs={
        "padding_side": "left",
    },
)
model.max_seq_length = 1024

In [ ]:
texts = chunks_df["chunk_text"].fillna("").astype(str).tolist()
BATCH_SIZE = 64
all_embeddings = []

for start in tqdm(
    range(0, len(texts), BATCH_SIZE),
    total=(len(texts) + BATCH_SIZE - 1) // BATCH_SIZE,
    desc="Embedding chunks",
    unit="batch",
):
    batch_texts = texts[start:start + BATCH_SIZE]

    batch_embeddings = model.encode(
        batch_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    all_embeddings.append(batch_embeddings)
    # del batch_embeddings
    # torch.cuda.empty_cache()

chunk_embeddings = np.vstack(all_embeddings)

print("Embedding matrix shape:", chunk_embeddings.shape)

In [ ]:
embeddings = []

for row, embedding in zip(chunks_df.itertuples(index=False), chunk_embeddings):
    embeddings.append({
        "chunk_id": row.chunk_id,
        "text": row.chunk_text,
        "document_id": row.document_id,
        "chunk_index": row.chunk_index,
        "embedding": embedding,
    })

embeddings_df = pd.DataFrame(embeddings)

In [ ]:
embeddings_df.to_parquet("rag_qa_ptbr_chunk_embeddings.parquet",index=False)

In [ ]:
from tqdm.auto import tqdm

questions_df = (
    df_split
    .sample(frac=0.05, random_state=42)
    [["document_id", "questions"]]
    .copy()
)

questions_df["question"] = (
    questions_df["questions"]
    .fillna("")
    .astype(str)
    .str.split(";")
    .str[0]
    .str.strip()
)

questions_df = questions_df.drop(columns=["questions"])
questions_df = questions_df[questions_df["question"] != ""].reset_index(drop=True)

In [ ]:
questions_df.to_parquet("data/sample_questions.parquet", index=False)